# Exploration Hacker News : choisir les mots-clés de collecte

**But** : trouver 4 à 6 mots-clés qui ramènent, via l'API Algolia de Hacker News, des **témoignages vécus** sur l'adoption de l'IA au travail, couvrant les 5 étapes ADKAR (Awareness, Desire, Knowledge, Ability, Reinforcement).

**Méthode en bref**
1. On interroge l'API pour 54 mots-clés (44 en première vague, 10 en deuxième), avec et sans guillemets, sur deux endpoints (`/search` = tri par pertinence, `/search_by_date` = tri par date).
2. Toutes les réponses sont gardées dans `data/exploration_hn_brut.json` : relancer le notebook ne réinterroge pas l'API.
3. Chaque commentaire unique est **lu et noté à la main** (0, 1 ou 2), à l'aveugle (sans savoir quel mot-clé l'a ramené). Les notes sont dans `data/notes_pertinence.json`.
4. On calcule les indicateurs par mot-clé, on choisit la meilleure **combinaison**, puis on vérifie sur 20 commentaires supplémentaires par finaliste.

Le fichier `data/` est ignoré par Git : pour reproduire, il faut relancer la collecte (cellule 3) une fois.

In [1]:
# Bibliothèques (toutes déjà installées dans le .venv)
import json, time, re, html, random
from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd

pd.set_option("display.max_colwidth", 80)
DOSSIER_DATA = Path("../data")
DOSSIER_DATA.mkdir(exist_ok=True)
FICHIER_BRUT = DOSSIER_DATA / "exploration_hn_brut.json"
FICHIER_NOTES = DOSSIER_DATA / "notes_pertinence.json"
URL_API = "https://hn.algolia.com/api/v1/"

## 1. Fonction d'appel à l'API (polie et avec cache)
- pause de 0,6 s avant chaque appel réseau (consigne : au moins 0,5 s) ;
- délai maximum de 15 s par requête ;
- si le code HTTP n'est pas 200 (ou erreur réseau), on réessaie jusqu'à 3 fois en attendant de plus en plus longtemps ;
- chaque réponse est rangée dans un dictionnaire `cache` sauvegardé sur disque. Si la même requête a déjà été faite, on relit le cache.

In [2]:
cache = json.loads(FICHIER_BRUT.read_text(encoding="utf-8")) if FICHIER_BRUT.exists() else {}
print(f"{len(cache)} réponses déjà en cache")

def cle_cache(endpoint, params):
    """Clé unique et stable pour une requête (endpoint + paramètres triés)."""
    return endpoint + "?" + "&".join(f"{k}={params[k]}" for k in sorted(params))

def interroger_api(endpoint, sauver=True, **params):
    """Appelle /search ou /search_by_date avec tags=comment et 50 résultats par défaut."""
    params = {"tags": "comment", "hitsPerPage": 50, **params}
    cle = cle_cache(endpoint, params)
    if cle in cache:
        return cache[cle]
    for essai in range(1, 4):
        time.sleep(0.6)  # pause de politesse
        try:
            r = requests.get(URL_API + endpoint, params=params, timeout=15)
            if r.status_code == 200:
                donnees = r.json()
                donnees["_collecte_le"] = datetime.now(timezone.utc).isoformat()
                cache[cle] = donnees
                if sauver:
                    FICHIER_BRUT.write_text(json.dumps(cache), encoding="utf-8")
                return donnees
            print(f"  code {r.status_code} pour {cle}, essai {essai}/3")
        except requests.RequestException as e:
            print(f"  erreur réseau ({e.__class__.__name__}) pour {cle}, essai {essai}/3")
        time.sleep(2 * essai)  # on attend plus longtemps avant de réessayer
    raise RuntimeError(f"Échec après 3 essais : {cle}")

def sauver_cache():
    FICHIER_BRUT.write_text(json.dumps(cache), encoding="utf-8")

383 réponses déjà en cache


## 2. Les mots-clés candidats
16 mots-clés imposés + nos propres idées, rangées selon l'étape ADKAR qu'on **espère** toucher (l'étape réelle sera mesurée à la lecture).
Chaque mot-clé est testé **sans guillemets** (l'API cherche les mots séparément) et **entre guillemets** (expression exacte).

In [3]:
CANDIDATS = {
    # --- les 16 mots-clés demandés ---
    "my manager AI": "imposé", "company mandated AI": "imposé", "management wants us to use AI": "imposé",
    "AI rollout at work": "imposé", "copilot company": "imposé", "forced to use copilot": "imposé",
    "mandatory AI usage": "imposé", "track our AI usage": "imposé", "leadership pushing AI": "imposé",
    "my employer AI": "imposé", "we were told to use AI": "imposé", "AI at my job": "imposé",
    "coworkers using AI": "imposé", "AI training at work": "imposé", "refuse to use AI at work": "imposé",
    "AI usage metrics": "imposé",
    # --- général : imposition / mandat ---
    "AI mandate": "général", "our CTO AI": "général", "pushed to use AI": "général",
    "my company AI tools": "général", "AI adoption at my company": "général",
    # --- Awareness : ne comprend pas pourquoi ---
    "AI for the sake of AI": "Awareness", "told to use AI for everything": "Awareness",
    "CEO wants AI everywhere": "Awareness", "don't understand why AI": "Awareness",
    # --- Desire : comprend mais ne veut pas ---
    "AI will replace my job": "Desire", "refuse to use copilot": "Desire",
    "I don't want to use AI": "Desire", "afraid AI replace me": "Desire", "AI layoffs at my company": "Desire",
    # --- Knowledge : veut mais ne sait pas comment ---
    "don't know how to use AI": "Knowledge", "no training AI tools": "Knowledge",
    "learning to use copilot": "Knowledge", "how to get value from AI at work": "Knowledge",
    # --- Ability : sait mais échoue en pratique ---
    "company banned ChatGPT": "Ability", "not allowed to use ChatGPT at work": "Ability",
    "security team AI tools": "Ability", "copilot useless on our codebase": "Ability", "legal won't let us use AI": "Ability",
    # --- Reinforcement : adopté puis abandonné ---
    "stopped using copilot": "Reinforcement", "cancelled copilot licenses": "Reinforcement",
    "gave up on AI tools": "Reinforcement", "nobody uses copilot": "Reinforcement",
    "went back to coding without AI": "Reinforcement",
}
# Variantes entre guillemets : l'expression entière, plus quelques guillemets partiels (expression courte + mot libre)
VARIANTES = {kw: [kw, f'"{kw}"'] for kw in CANDIDATS}
VARIANTES["my manager AI"].append('"my manager" AI')
VARIANTES["my employer AI"].append('"my employer" AI')
VARIANTES["AI at my job"].append('"at my job" AI')
VARIANTES["my company AI tools"].append('"my company" AI')
VARIANTES["copilot company"].append('copilot "my company"')
VARIANTES["coworkers using AI"].append('"my coworkers" AI')
REQUETES = [(kw, q) for kw, qs in VARIANTES.items() for q in qs]
ENDPOINTS = ["search", "search_by_date"]
print(f"{len(CANDIDATS)} mots-clés, {len(REQUETES)} requêtes x {len(ENDPOINTS)} endpoints = {len(REQUETES)*len(ENDPOINTS)} appels")

44 mots-clés, 94 requêtes x 2 endpoints = 188 appels


## 3. Collecte (première page de 50 résultats par requête et par endpoint)
Environ 3 minutes la première fois, instantané ensuite grâce au cache.

In [4]:
for kw, q in REQUETES:
    for ep in ENDPOINTS:
        interroger_api(ep, sauver=False, query=q)
sauver_cache()
print(f"{len(cache)} réponses en cache, fichier de {FICHIER_BRUT.stat().st_size/1e6:.1f} Mo")

383 réponses en cache, fichier de 26.4 Mo


## 4. Deuxième vague : variantes d'« ancrage à la première personne »
La première lecture (section 6) a montré que les meilleurs résultats viennent d'une **expression personnelle entre guillemets + AI** (`"my manager" AI`, `"my company" AI`…), et que les mots-clés « par étape ADKAR » échouent. On remet donc ce choix en cause en testant d'autres ancrages du même type.

In [5]:
VAGUE2 = ['"my boss" AI', '"my team" AI', '"our CEO" AI', '"my org" AI', '"at work" AI',
          '"my workplace" AI', '"my colleagues" AI', '"our company" AI', '"my job" AI', '"my company" Claude']
for q in VAGUE2:
    for ep in ENDPOINTS:
        interroger_api(ep, sauver=False, query=q)
sauver_cache()
for q in VAGUE2:
    CANDIDATS[q] = "vague 2"; VARIANTES[q] = [q]
REQUETES = [(kw, q) for kw, qs in VARIANTES.items() for q in qs]
print(f"{len(REQUETES)} requêtes au total, {len(cache)} réponses en cache")

104 requêtes au total, 383 réponses en cache


## 5. Tests de l'API : guillemets, champs recherchés, fautes de frappe, filtres
Questions vérifiées :
1. Les guillemets sont-ils respectés ? (on compte la présence exacte de l'expression dans le texte)
2. La recherche porte-t-elle aussi sur le **titre du fil** ? (`restrictSearchableAttributes=comment_text` pour l'empêcher)
3. La tolérance aux fautes gonfle-t-elle les volumes ? (`typoTolerance=false`)
4. Le filtre de date `numericFilters=created_at_i>...` fonctionne-t-il ?

In [6]:
def texte_propre(h):
    """Texte du commentaire sans HTML, sans entités (&#x27;) et sans citations (lignes commençant par >)."""
    t = html.unescape(h.get("comment_text") or "")
    t = re.sub(r"<p>", "\n", t)
    t = re.sub(r"<[^>]+>", "", t)
    t = "\n".join(l for l in t.split("\n") if not l.strip().startswith(">"))
    return re.sub(r"\s+", " ", t).strip()

def expression_exacte(q):
    """Partie entre guillemets d'une requête, ou None."""
    m = re.search(r'"([^"]+)"', q)
    return m.group(1).lower() if m else None

lignes = []
for kw, q in REQUETES:
    phr = expression_exacte(q)
    if not phr:
        continue
    d = interroger_api("search", query=q)
    if not d["hits"]:
        continue
    brut = [html.unescape(re.sub(r"<[^>]+>", " ", h.get("comment_text") or "")).lower() for h in d["hits"]]
    # apostrophes typographiques normalisées
    brut = [t.replace("’", "'") for t in brut]
    lignes.append({"requête": q, "résultats": len(brut),
                   "% contenant l'expression exacte": round(100 * sum(phr in t for t in brut) / len(brut))})
pd.DataFrame(lignes).sort_values("% contenant l'expression exacte").head(8)

,requête,résultats,% contenant l'expression exacte
1,"""company mandated AI""",1,0
21,"""I don't want to use AI""",10,70
13,"""AI mandate""",22,82
8,"""AI at my job""",10,90
16,"""AI for the sake of AI""",12,92
11,"""my coworkers"" AI",50,94
18,"""don't understand why AI""",25,96
32,"""my colleagues"" AI",50,96


In [7]:
# Comparaison des volumes (nbHits) selon les paramètres, sur les requêtes les plus prometteuses
TESTS = ['"my manager" AI', '"my company" AI', 'copilot "my company"', '"my coworkers" AI',
         '"at my job" AI', '"my employer" AI', 'my manager AI']
DATE_REF = int(datetime(2026, 11, 24, tzinfo=timezone.utc).timestamp())  # date de collecte
UN_AN = DATE_REF - 365 * 86400
lignes = []
for q in TESTS:
    base = interroger_api("search", query=q, hitsPerPage=0)["nbHits"]
    base_date = interroger_api("search_by_date", query=q, hitsPerPage=0)["nbHits"]
    restreint = interroger_api("search", query=q, hitsPerPage=0, restrictSearchableAttributes="comment_text")["nbHits"]
    sans_typo = interroger_api("search", query=q, hitsPerPage=0, typoTolerance="false")["nbHits"]
    recents = interroger_api("search", query=q, hitsPerPage=0, numericFilters=f"created_at_i>{UN_AN}")["nbHits"]
    lignes.append({"requête": q, "/search": base, "/search_by_date": base_date,
                   "texte seul": restreint, "sans fautes": sans_typo, "12 derniers mois": recents})
sauver_cache()
pd.DataFrame(lignes)

,requête,/search,/search_by_date,texte seul,sans fautes,12 derniers mois
0,"""my manager"" AI",314,314,298,314,102
1,"""my company"" AI",2720,3197,2665,2777,724
2,"copilot ""my company""",170,170,170,168,57
3,"""my coworkers"" AI",519,519,505,519,155
4,"""at my job"" AI",345,363,328,317,116
5,"""my employer"" AI",824,824,811,824,231
6,my manager AI,10487,14185,10086,5693,1985


## 6. Notation manuelle (faite hors notebook, résultat chargé ici)
Chaque commentaire unique a été **lu** et noté à l'aveugle (ordre mélangé, mot-clé masqué, pseudo masqué) :
- **2** = témoignage personnel riche sur l'usage / l'imposition / le refus / la formation / l'abandon d'un outil d'IA au travail ;
- **1** = témoignage partiel (court, rapporté, ou expérience pro peu détaillée) ;
- **0** = hors sujet (débat général, technique, économie, projet perso, offre d'emploi, < 150 caractères).

Pour chaque note > 0, on a aussi noté la ou les étapes ADKAR visibles : AW (Awareness), DE (Desire), KN (Knowledge), AB (Ability), RE (Reinforcement), ou « - » si c'est une adoption sans frein.
Les < 150 caractères et les fils « Who is hiring / Seeking work » ont reçu 0 automatiquement (règle de la consigne).

In [8]:
notes = json.loads(FICHIER_NOTES.read_text(encoding="utf-8"))
repartition = pd.Series([v["note"] for v in notes.values()]).value_counts().sort_index()
print(f"{len(notes)} commentaires uniques notés")
print("Répartition des notes :", repartition.to_dict())
etapes = pd.Series([e for v in notes.values() if v["note"] > 0 for e in v["etapes"]]).value_counts()
print("Étapes ADKAR repérées (commentaires pertinents) :", etapes.to_dict())

4858 commentaires uniques notés
Répartition des notes : {0: 3801, 1: 746, 2: 311}
Étapes ADKAR repérées (commentaires pertinents) : {'AB': 382, 'AW': 284, 'DE': 217, 'KN': 115, 'RE': 85}


## 7. Tableau des indicateurs par mot-clé et par endpoint
- **volume** = `nbHits` (total annoncé par l'API) ;
- **score /100** = somme des notes des 50 premiers résultats (max 2 × 50 = 100). Si l'API renvoie moins de 50 résultats, le score est ramené sur 100 ;
- **% pertinents** = part des résultats notés 1 ou 2 ;
- **% récents** = part publiée dans les 12 mois avant la collecte (24/11/2026) ;
- **% doublons** = part des résultats qu'on retrouve aussi dans un *autre* mot-clé (via `objectID`).

In [9]:
resultats = {}  # (requête, endpoint) -> liste des hits
for kw, q in REQUETES:
    for ep in ENDPOINTS:
        resultats[(q, ep)] = interroger_api(ep, query=q)["hits"]

# pour les doublons : dans quels mots-clés apparaît chaque objectID ?
mots_par_id = {}
for (q, ep), hits in resultats.items():
    for h in hits:
        mots_par_id.setdefault(h["objectID"], set()).add(q)

lignes = []
for (q, ep), hits in resultats.items():
    n = len(hits)
    if n == 0:
        lignes.append({"requête": q, "endpoint": ep, "volume": 0, "n lus": 0})
        continue
    ns = [notes[h["objectID"]]["note"] for h in hits]
    et = sorted({e for h in hits if notes[h["objectID"]]["note"] > 0 for e in notes[h["objectID"]]["etapes"] if e != "-"})
    lignes.append({
        "requête": q, "endpoint": ep,
        "volume": cache[cle_cache(ep, {"tags": "comment", "hitsPerPage": 50, "query": q})]["nbHits"],
        "n lus": n,
        "score /100": round(100 * sum(ns) / (2 * n)),
        "% pertinents": round(100 * sum(x > 0 for x in ns) / n),
        "% récents": round(100 * sum(h["created_at_i"] >= UN_AN for h in hits) / n),
        "longueur moy.": round(sum(len(texte_propre(h)) for h in hits) / n),
        "% doublons": round(100 * sum(len(mots_par_id[h["objectID"]]) > 1 for h in hits) / n),
        "étapes ADKAR": " ".join(et),
    })
tableau = pd.DataFrame(lignes).sort_values("score /100", ascending=False)
print(f"{len(tableau)} combinaisons requête x endpoint")
tableau.head(25)

208 combinaisons requête x endpoint


,requête,endpoint,volume,n lus,score /100,% pertinents,% récents,longueur moy.,% doublons,étapes ADKAR
8,"""company mandated AI""",search,1,1,50.0,100.0,100.0,1285.0,100.0,AW
9,"""company mandated AI""",search_by_date,1,1,50.0,100.0,100.0,1285.0,100.0,AW
103,"""told to use AI for everything""",search_by_date,2,2,50.0,50.0,100.0,219.0,100.0,AW
102,"""told to use AI for everything""",search,2,2,50.0,50.0,100.0,219.0,100.0,AW
161,copilot useless on our codebase,search_by_date,1,1,50.0,100.0,100.0,245.0,0.0,AB
160,copilot useless on our codebase,search,1,1,50.0,100.0,100.0,245.0,0.0,AB
5,"""my manager"" AI",search_by_date,314,50,46.0,62.0,100.0,936.0,36.0,AB AW DE KN RE
23,"copilot ""my company""",search_by_date,170,50,45.0,64.0,100.0,787.0,34.0,AB AW DE KN RE
52,"""AI at my job""",search,10,10,45.0,70.0,80.0,458.0,100.0,AB AW DE
53,"""AI at my job""",search_by_date,10,10,45.0,70.0,80.0,458.0,100.0,AB AW DE


In [10]:
# Ce qui ne marche pas : les mots-clés « par étape ADKAR » (sans guillemets) et les phrases entières entre guillemets
par_categorie = []
for kw, q in REQUETES:
    for ep in ENDPOINTS:
        ligne = tableau[(tableau["requête"] == q) & (tableau["endpoint"] == ep)].iloc[0]
        if ligne["n lus"] == 0:
            continue
        type_req = ("ancrage perso entre guillemets" if re.fullmatch(r'"(my|our|at my|at) [^"]+" \w+|\w+ "my [^"]+"', q)
                    else "phrase entière entre guillemets" if q.startswith('"') and q.endswith('"')
                    else "mots libres")
        par_categorie.append({"type": type_req, "groupe": CANDIDATS.get(kw, ""), "endpoint": ep,
                              "score": ligne["score /100"], "volume": ligne["volume"]})
pc = pd.DataFrame(par_categorie)
pc.groupby(["type", "endpoint"]).agg(requetes=("score", "size"), score_moyen=("score", "mean"),
                                    volume_median=("volume", "median")).round(0)

requetes  score_moyen  \
type                            endpoint                                
ancrage perso entre guillemets  search                16         29.0   
                                search_by_date        16         31.0   
mots libres                     search                44         14.0   
                                search_by_date        44          8.0   
phrase entière entre guillemets search                20         15.0   
                                search_by_date        20         15.0   

                                                volume_median  
type                            endpoint                       
ancrage perso entre guillemets  search                  580.0  
                                search_by_date          580.0  
mots libres                     search                  544.0  
                                search_by_date          695.0  
phrase entière entre guillemets search                    6.0  
                                search_by_date            6.0

## 8. Choix de la combinaison finale
Critères, dans cet ordre :
1. **précision mesurée sur `/search_by_date`** : cet endpoint renvoie les 50 plus récents, donc un échantillon représentatif de ce que l'ETL va réellement télécharger (alors que `/search` renvoie les 50 « meilleurs », trop optimistes pour la suite) ;
2. **volume** suffisant pour viser ~2 000 verbatims ;
3. **couverture des 5 étapes ADKAR** ;
4. **faible chevauchement** entre mots-clés (mesuré ci-dessous par des requêtes combinées).

In [11]:
FINALISTES = ['"my manager" AI', '"my team" AI', '"my company" AI', '"my coworkers" AI',
              '"at my job" AI', '"my workplace" AI', '"my org" AI', '"my employer" AI', 'copilot "my company"']
ancre = lambda q: expression_exacte(q)
lignes = []
for a in FINALISTES:
    for b in FINALISTES:
        if a >= b:
            continue
        mot_b = "copilot" if b.startswith("copilot") else "AI"
        mot_a = "copilot" if a.startswith("copilot") else "AI"
        if mot_a != mot_b:
            continue
        q = f'"{ancre(a)}" "{ancre(b)}" {mot_a}'
        inter = interroger_api("search_by_date", query=q, hitsPerPage=0)["nbHits"]
        lignes.append({"A": a, "B": b, "A et B": inter})
sauver_cache()
chev = pd.DataFrame(lignes)
vol = {q: cache[cle_cache("search_by_date", {"tags": "comment", "hitsPerPage": 50, "query": q})]["nbHits"] for q in FINALISTES}
chev["% de A"] = (100 * chev["A et B"] / chev["A"].map(vol)).round(1)
chev.sort_values("A et B", ascending=False).head(8)

,A,B,A et B,% de A
5,"""my company"" AI","""my team"" AI",30,1.0
9,"""my company"" AI","""my employer"" AI",14,0.5
0,"""my manager"" AI","""my team"" AI",13,4.1
6,"""my company"" AI","""my coworkers"" AI",12,0.4
17,"""at my job"" AI","""my company"" AI",8,2.2
11,"""my coworkers"" AI","""my team"" AI",7,1.3
16,"""at my job"" AI","""my team"" AI",7,1.9
10,"""my coworkers"" AI","""my manager"" AI",7,1.3


## 9. Deuxième passe : 20 commentaires de plus par finaliste
On récupère la page suivante de `/search_by_date` (rangs 51 à 70) et on lit 20 commentaires de plus par mot-clé, toujours à l'aveugle.
But : vérifier que la précision mesurée sur les 50 premiers n'était pas un coup de chance.

In [12]:
PASSE2 = ['"my manager" AI', '"my team" AI', '"my company" AI', '"my coworkers" AI',
          '"at my job" AI', '"my workplace" AI', '"my org" AI']
passe2 = {}
for q in PASSE2:
    hits = interroger_api("search_by_date", query=q, page=1)["hits"][:20]
    passe2[q] = hits
sauver_cache()
print({q: len(h) for q, h in passe2.items()})

{'"my manager" AI': 20, '"my team" AI': 20, '"my company" AI': 20, '"my coworkers" AI': 20, '"at my job" AI': 20, '"my workplace" AI': 20, '"my org" AI': 20}


In [13]:
# Note de la deuxième passe (sur 20, ramenée sur 100) comparée à la première (sur 50)
lignes = []
for q in PASSE2:
    n2 = [notes[h["objectID"]]["note"] for h in passe2[q]]
    n1 = [notes[h["objectID"]]["note"] for h in resultats[(q, "search_by_date")]]
    lignes.append({"requête": q,
                   "passe 1 : score /100": round(100 * sum(n1) / (2 * len(n1))),
                   "passe 1 : % pertinents": round(100 * sum(x > 0 for x in n1) / len(n1)),
                   "passe 2 : score /100": round(100 * sum(n2) / (2 * len(n2))),
                   "passe 2 : % pertinents": round(100 * sum(x > 0 for x in n2) / len(n2)),
                   "cumul (70) : % pertinents": round(100 * sum(x > 0 for x in n1 + n2) / len(n1 + n2))})
confirmation = pd.DataFrame(lignes).set_index("requête")
confirmation

,passe 1 : score /100,passe 1 : % pertinents,passe 2 : score /100,passe 2 : % pertinents,cumul (70) : % pertinents
requête,,,,,
"""my manager"" AI",46,62,38,55,60
"""my team"" AI",36,50,25,40,47
"""my company"" AI",29,44,28,40,43
"""my coworkers"" AI",35,56,25,45,53
"""at my job"" AI",35,62,50,70,64
"""my workplace"" AI",45,70,32,50,64
"""my org"" AI",36,58,40,60,59


## 10. Volume réel disponible (et pas seulement `nbHits`)
**Piège découvert** : `nbHits` est parfois une *estimation* (`exhaustive.nbHits = False`) qui varie d'un appel à l'autre (ex. `"my org" AI` : 305 puis 194). On compte donc les commentaires **réellement téléchargeables** en paginant.
L'API ne renvoie jamais plus de 1 000 résultats par requête : on découpe donc par **tranches de 6 mois** avec `numericFilters`. Pour rester léger, on ne demande que l'identifiant, la date et le titre du fil (`attributesToRetrieve`).
On ne garde que l'ère « ChatGPT » : à partir du 1er janvier 2023 (avant, « AI » désigne surtout autre chose).

In [14]:
DEBUT = int(datetime(2023, 1, 1, tzinfo=timezone.utc).timestamp())
SIX_MOIS = 182 * 86400
def tous_les_ids(q):
    """Tous les commentaires (id, date, titre) d'une requête depuis DEBUT, par tranches de 6 mois."""
    trouves = {}
    debut = DEBUT
    while debut < DATE_REF:
        fin = min(debut + SIX_MOIS, DATE_REF)
        page = 0
        while True:
            d = interroger_api("search_by_date", sauver=False, query=q, hitsPerPage=1000, page=page,
                               numericFilters=f"created_at_i>={debut},created_at_i<{fin}",
                               restrictSearchableAttributes="comment_text",
                               attributesToRetrieve="objectID,created_at_i,story_title",
                               attributesToHighlight="none")
            for h in d["hits"]:
                trouves[h["objectID"]] = h
            page += 1
            if page >= d["nbPages"]:
                break
        debut = fin
    return trouves

volumes = {q: tous_les_ids(q) for q in PASSE2}
sauver_cache()
SEMAINE = 7 * 86400
lignes = []
for q, ids in volumes.items():
    dates = [h["created_at_i"] for h in ids.values()]
    lignes.append({"requête": q, "depuis 2023": len(ids),
                   "12 derniers mois": sum(t >= UN_AN for t in dates),
                   "par semaine (moy. 12 dernières sem.)": round(sum(t >= DATE_REF - 12 * SEMAINE for t in dates) / 12, 1)})
vol_reel = pd.DataFrame(lignes).set_index("requête")
vol_reel

,depuis 2023,12 derniers mois,par semaine (moy. 12 dernières sem.)
requête,,,
"""my manager"" AI",186,102,1.7
"""my team"" AI",1022,552,11.6
"""my company"" AI",1470,721,15.9
"""my coworkers"" AI",285,154,3.2
"""at my job"" AI",187,116,2.1
"""my workplace"" AI",130,68,1.2
"""my org"" AI",170,101,1.5


In [15]:
# Mêmes mesures pour des ancrages "de réserve" (précision mesurée sur 50 seulement)
RESERVE = ['"my employer" AI', '"my boss" AI', '"my job" AI', '"our company" AI', '"our CEO" AI']
for q in RESERVE:
    volumes[q] = tous_les_ids(q)
sauver_cache()
for q in RESERVE:
    dates = [h["created_at_i"] for h in volumes[q].values()]
    vol_reel.loc[q] = [len(dates), sum(t >= UN_AN for t in dates),
                       round(sum(t >= DATE_REF - 12 * SEMAINE for t in dates) / 12, 1)]
vol_reel

,depuis 2023,12 derniers mois,par semaine (moy. 12 dernières sem.)
requête,,,
"""my manager"" AI",186,102,1.7
"""my team"" AI",1022,552,11.6
"""my company"" AI",1470,721,15.9
"""my coworkers"" AI",285,154,3.2
"""at my job"" AI",187,116,2.1
"""my workplace"" AI",130,68,1.2
"""my org"" AI",170,101,1.5
"""my employer"" AI",466,231,4.1
"""my boss"" AI",366,192,3.6


### 10 bis. Vérifications avant le choix final
- deuxième passe (20 de plus) aussi pour `"my job" AI` et `"my employer" AI`, candidats pour le volume ;
- **stabilité dans le temps** : la précision a été mesurée sur des commentaires de 2026. Or la moitié du volume date de 2023-2025. On lit donc 10 commentaires de **2023-2024** par mot-clé candidat.

In [16]:
for q in ['"my job" AI', '"my employer" AI']:
    passe2[q] = interroger_api("search_by_date", query=q, page=1)["hits"][:20]
FIN_2024 = int(datetime(2025, 1, 1, tzinfo=timezone.utc).timestamp())
CANDIDATS_FINAUX = ['"my company" AI', '"my team" AI', '"my job" AI', '"my coworkers" AI',
                    '"my manager" AI', '"my employer" AI', '"my org" AI']
anciens = {q: interroger_api("search_by_date", query=q, hitsPerPage=10,
                             numericFilters=f"created_at_i>={DEBUT},created_at_i<{FIN_2024}")["hits"]
           for q in CANDIDATS_FINAUX}
sauver_cache()
print({q: len(h) for q, h in anciens.items()})

{'"my company" AI': 10, '"my team" AI': 10, '"my job" AI': 10, '"my coworkers" AI': 10, '"my manager" AI': 10, '"my employer" AI': 10, '"my org" AI': 10}


In [17]:
# Précision par période : 2026 (passes 1 + 2) contre 2023-2024 (10 commentaires par mot-clé)
lignes = []
for q in CANDIDATS_FINAUX:
    recents = resultats[(q, "search_by_date")] + passe2[q]
    p_rec = sum(notes[h["objectID"]]["note"] > 0 for h in recents) / len(recents)
    p_old = sum(notes[h["objectID"]]["note"] > 0 for h in anciens[q]) / len(anciens[q])
    lignes.append({"requête": q, "lus (récents)": len(recents), "% pertinents récents": round(100 * p_rec),
                   "lus (2023-24)": len(anciens[q]), "% pertinents 2023-24": round(100 * p_old)})
stabilite = pd.DataFrame(lignes).set_index("requête")
N_ANCIENS = sum(len(anciens[q]) for q in CANDIDATS_FINAUX)
P_ANCIEN = sum(notes[h["objectID"]]["note"] > 0 for q in CANDIDATS_FINAUX for h in anciens[q]) / N_ANCIENS
print(f"Précision moyenne sur 2023-2024 : {P_ANCIEN:.0%} (échantillon de {N_ANCIENS} commentaires)")
stabilite

Précision moyenne sur 2023-2024 : 19% (échantillon de 70 commentaires)


,lus (récents),% pertinents récents,lus (2023-24),% pertinents 2023-24
requête,,,,
"""my company"" AI",70,43,10,40
"""my team"" AI",70,47,10,10
"""my job"" AI",70,39,10,10
"""my coworkers"" AI",70,53,10,30
"""my manager"" AI",70,60,10,10
"""my employer"" AI",70,40,10,0
"""my org"" AI",70,59,10,30


**Lecture** : la précision chute nettement sur 2023-2024 (environ 1 commentaire pertinent sur 5). Les témoignages d'IA *imposée au travail* sont surtout un phénomène récent (2025-2026). Le volume utile est donc estimé **par période** : précision récente sur les 12 derniers mois, précision 2023-2024 avant. L'échantillon ancien est petit (10 par mot-clé) : on utilise la moyenne commune plutôt que les valeurs par mot-clé.

## 11. Combinaison retenue et volume attendu
Règle de sélection (6 mots-clés maximum) :
- précision récente d'au moins 39 % sur 70 commentaires lus (passe 1 + passe 2) ;
- maximiser le nombre estimé de commentaires **pertinents** depuis 2023 ;
- éviter les doublons logiques : `"at my job" AI` est inclus dans `"my job" AI`, donc écarté ;
- à volume proche, garder le plus précis (`"my manager" AI` plutôt que `"my org" AI` ou `"my workplace" AI`).

In [18]:
CHOIX = ['"my job" AI', '"my company" AI', '"my team" AI', '"my employer" AI', '"my coworkers" AI', '"my manager" AI']
lignes = []
for q in CHOIX:
    ids = volumes[q]
    recents = sum(h["created_at_i"] >= UN_AN for h in ids.values())
    anciens_n = len(ids) - recents
    p_rec = stabilite.loc[q, "% pertinents récents"] / 100
    sem = sum(h["created_at_i"] >= DATE_REF - 12 * SEMAINE for h in ids.values()) / 12
    lignes.append({"requête": q, "bruts depuis 2023": len(ids), "dont 12 derniers mois": recents,
                   "% pertinents récents": round(100 * p_rec),
                   "pertinents estimés": round(recents * p_rec + anciens_n * P_ANCIEN),
                   "bruts / semaine": round(sem, 1), "pertinents / semaine": round(sem * p_rec, 1)})
choix = pd.DataFrame(lignes).set_index("requête")
union = {}
for q in CHOIX:
    union.update(volumes[q])
doublons = sum(len(volumes[q]) for q in CHOIX) - len(union)
PERTINENTS_ESTIMES = choix["pertinents estimés"].sum() * len(union) / choix["bruts depuis 2023"].sum()
print(f"Union des 6 mots-clés : {len(union)} commentaires bruts uniques depuis 2023 "
      f"({doublons} doublons retirés, soit {100 * doublons / len(union):.1f} %)")
print(f"Pertinents estimés : environ {PERTINENTS_ESTIMES:.0f} (incertitude forte, de l'ordre de ±25 %)")
print(f"Flux actuel : environ {choix['bruts / semaine'].sum():.0f} bruts par semaine, "
      f"dont environ {choix['pertinents / semaine'].sum():.0f} pertinents")
choix

Union des 6 mots-clés : 5416 commentaires bruts uniques depuis 2023 (182 doublons retirés, soit 3.4 %)
Pertinents estimés : environ 1675 (incertitude forte, de l'ordre de ±25 %)
Flux actuel : environ 57 bruts par semaine, dont environ 25 pertinents


,bruts depuis 2023,dont 12 derniers mois,% pertinents récents,pertinents estimés,bruts / semaine,pertinents / semaine
requête,,,,,,
"""my job"" AI",2169,1045,39,616,20.8,8.1
"""my company"" AI",1470,721,43,449,15.9,6.8
"""my team"" AI",1022,552,47,347,11.6,5.4
"""my employer"" AI",466,231,40,136,4.1,1.6
"""my coworkers"" AI",285,154,53,106,3.2,1.7
"""my manager"" AI",186,102,60,77,1.7,1.0


In [19]:
# Couverture ADKAR de chaque mot-clé retenu (sur les 70 commentaires récents lus)
lignes = []
for q in CHOIX:
    c_et = pd.Series([e for h in resultats[(q, "search_by_date")] + passe2[q]
                      if notes[h["objectID"]]["note"] > 0 for e in notes[h["objectID"]]["etapes"]]).value_counts()
    lignes.append({"requête": q, **{e: int(c_et.get(e, 0)) for e in ["AW", "DE", "KN", "AB", "RE"]}})
couverture = pd.DataFrame(lignes).set_index("requête")
couverture.loc["TOTAL"] = couverture.sum()
couverture

,AW,DE,KN,AB,RE
requête,,,,,
"""my job"" AI",7,12,2,9,0
"""my company"" AI",8,7,2,11,3
"""my team"" AI",4,7,6,10,2
"""my employer"" AI",10,8,1,12,5
"""my coworkers"" AI",2,7,6,24,0
"""my manager"" AI",18,13,2,18,2
TOTAL,49,54,19,84,12


## 12. Problèmes de données à traiter au nettoyage (mesurés sur les commentaires lus)

In [20]:
textes = {}
for (q, ep), hits in resultats.items():
    for h in hits:
        textes[h["objectID"]] = h.get("comment_text") or ""
brut = list(textes.values())
def part(motif):
    return f"{100 * sum(bool(re.search(motif, t)) for t in brut) / len(brut):.0f} %"
problemes = pd.DataFrame([
    {"problème": "balises HTML <p>", "part des commentaires": part(r"<p>")},
    {"problème": "liens <a href=...>", "part des commentaires": part(r"<a href")},
    {"problème": "italique <i> ou code <pre><code>", "part des commentaires": part(r"<i>|<pre>|<code>")},
    {"problème": "entités HTML (&#x27; &quot; &gt; &amp; &#x2F;)", "part des commentaires": part(r"&#x27;|&quot;|&gt;|&amp;|&#x2F;")},
    {"problème": "citation d'un autre commentaire (ligne qui commence par &gt;)", "part des commentaires": part(r"(^|<p>)\s*&gt;")},
    {"problème": "adresses web en clair", "part des commentaires": part(r"https?://")},
    {"problème": "mention d'un pseudo (@xxx, « GP », « parent comment »)", "part des commentaires": part(r"(?<![\w.])@\w+|\bGP\b|\bparent comment\b")},
])
vus = {}
for oid, t in textes.items():
    vus.setdefault(re.sub(r"\W+", "", html.unescape(t)).lower()[:300], []).append(oid)
print("Textes identiques publiés sous deux objectID différents :", sum(len(v) > 1 for v in vus.values()))
print("Commentaires de moins de 150 caractères après nettoyage :",
      sum(len(texte_propre({"comment_text": t})) < 150 for t in brut), "sur", len(brut))
problemes

Textes identiques publiés sous deux objectID différents : 19
Commentaires de moins de 150 caractères après nettoyage : 198 sur 4458


,problème,part des commentaires
0,balises HTML <p>,85 %
1,liens <a href=...>,14 %
2,italique <i> ou code <pre><code>,19 %
3,entités HTML (&#x27; &quot; &gt; &amp; &#x2F;),84 %
4,citation d'un autre commentaire (ligne qui commence par &gt;),18 %
5,adresses web en clair,1 %
6,"mention d'un pseudo (@xxx, « GP », « parent comment »)",1 %


In [21]:
# Trois extraits représentatifs (note 2) par mot-clé retenu : sans pseudo, nettoyés et tronqués
random.seed(7)
for q in CHOIX:
    riches = [h for h in resultats[(q, "search_by_date")] + passe2[q] if notes[h["objectID"]]["note"] == 2]
    print(f"\n=== {q} ({len(riches)} commentaires notés 2 sur 70) ===")
    for h in random.sample(riches, min(3, len(riches))):
        txt = texte_propre(h)
        i = max(0, txt.lower().find(expression_exacte(q)) - 100)   # fenêtre centrée sur l'ancrage
        print(f"- [{'/'.join(notes[h['objectID']]['etapes'])}] …{txt[i:i + 300]}…")


=== "my job" AI (5 commentaires notés 2 sur 70) ===
- [AW/AB] …agement has decided to trust AI reviews, and we ship to production without a human in the loop. Now my job is to manage tokens and invent new ways of AI-guided testing of already shipped code after release. It's better than review fatigue, but I still feel bad.…
- [KN/AB] …For a couple of year I have been reviewing code in my team and my job is only to review and coach other junior/intermediate members of the team - not to develop new features myself. The problem with AI is the merge requests now contain more code and often the chosen solution is overblown/overdesigne…
- [AW/DE] …with AI, but the we’re wasting so much time doing a ton of extra things very efficiently. Let me do my job. My manager should know how I’m doing. Maybe highlight key wins. CSATs/etc go out every year.…

=== "my company" AI (10 commentaires notés 2 sur 70) ===
- [AW/DE] …I have been raising this point in my company (which targets almost zero human 

In [22]:
# Tableau récapitulatif compact (endpoint /search_by_date, précision sur 50 lus, ou 70 si deuxième passe)
A_AFFICHER = CHOIX + ['"my org" AI', '"my workplace" AI', '"at my job" AI', '"my boss" AI', '"our company" AI',
                      'copilot "my company"', 'my manager AI', 'forced to use copilot', '"forced to use copilot"',
                      'AI mandate', 'leadership pushing AI', 'AI will replace my job', "don't know how to use AI",
                      'stopped using copilot', 'company banned ChatGPT']
ids_choix = {q: set(volumes[q]) for q in CHOIX}
lignes = []
for q in A_AFFICHER:
    hits = resultats[(q, "search_by_date")] + passe2.get(q, [])
    ns = [notes[h["objectID"]]["note"] for h in hits]
    et = sorted({e for h in hits if notes[h["objectID"]]["note"] > 0 for e in notes[h["objectID"]]["etapes"] if e != "-"})
    if q in volumes:
        autres = set().union(*[ids_choix[o] for o in CHOIX if o != q])
        vol = len(volumes[q]); dbl = round(100 * len(set(volumes[q]) & autres) / max(1, vol))
        rec = round(100 * sum(h["created_at_i"] >= UN_AN for h in volumes[q].values()) / max(1, vol))
    else:
        vol = cache[cle_cache("search_by_date", {"tags": "comment", "hitsPerPage": 50, "query": q})]["nbHits"]
        dbl = None; rec = round(100 * sum(h["created_at_i"] >= UN_AN for h in hits) / max(1, len(hits)))
    lignes.append({"mot-clé": q, "volume": vol, "lus": len(hits), "score /100": round(100 * sum(ns) / (2 * len(ns))),
                   "% récents": rec, "% doublons": dbl, "étapes": " ".join(et), "retenu": "oui" if q in CHOIX else "non"})
recap = pd.DataFrame(lignes).set_index("mot-clé")
recap

,volume,lus,score /100,% récents,% doublons,étapes,retenu
mot-clé,,,,,,,
"""my job"" AI",2169,70,23,48,6.0,AB AW DE KN,oui
"""my company"" AI",1470,70,29,49,6.0,AB AW DE KN RE,oui
"""my team"" AI",1022,70,33,54,6.0,AB AW DE KN RE,oui
"""my employer"" AI",466,70,29,50,8.0,AB AW DE KN RE,oui
"""my coworkers"" AI",285,70,32,54,8.0,AB AW DE KN,oui
"""my manager"" AI",186,70,44,55,12.0,AB AW DE KN RE,oui
"""my org"" AI",170,70,37,59,7.0,AB AW DE KN RE,non
"""my workplace"" AI",130,70,41,52,6.0,AB AW DE KN RE,non
"""at my job"" AI",187,70,39,62,100.0,AB AW DE,non


## 13. Ce que l'ETL réutilise

In [23]:
# --- Paramètres de collecte retenus (à copier dans l'ETL) ---
MOTS_CLES = [
    '"my job" AI',
    '"my company" AI',
    '"my team" AI',
    '"my employer" AI',
    '"my coworkers" AI',
    '"my manager" AI',
]
ENDPOINT = "search_by_date"  # échantillon représentatif + collecte incrémentale par date
FILTRES = {
    # paramètres envoyés à l'API
    "tags": "comment",
    "hitsPerPage": 1000,                              # maximum autorisé par page
    "restrictSearchableAttributes": "comment_text",   # ne pas matcher le titre du fil
    "numericFilters_debut": DEBUT,                    # created_at_i >= 01/01/2023 (ère ChatGPT)
    "tranche_secondes": SIX_MOIS,                     # découper par 6 mois : l'API plafonne à 1000 résultats par requête
    "pause_secondes": 0.6, "timeout_secondes": 15, "essais_max": 3,
    # filtres appliqués après téléchargement (côté Python)
    "longueur_min_caracteres": 150,                   # après retrait du HTML, des entités et des citations "&gt;"
    "exclure_titres_regex": r"who is hiring|seeking work|who wants to be hired|freelancer\? seeking",
    "dedoublonner_sur": "objectID",
}
print(MOTS_CLES, ENDPOINT, sep="\n")

['"my job" AI', '"my company" AI', '"my team" AI', '"my employer" AI', '"my coworkers" AI', '"my manager" AI']
search_by_date


## 14. Synthèse

### Tableau récapitulatif (endpoint `/search_by_date`)
Volume = commentaires réellement téléchargeables depuis le 01/01/2023 pour les mots-clés mesurés en détail, sinon `nbHits` annoncé (souvent surestimé). Score et étapes ADKAR : sur 70 commentaires lus (deux passes) pour les 9 finalistes, sur 50 ou moins pour les autres. % récents = part des 12 derniers mois dans le volume depuis 2023 (finalistes), ou dans l'échantillon lu (autres). % doublons = part retrouvée dans un autre mot-clé retenu. Le tableau complet (208 combinaisons) est en section 7.

| mot-clé | volume | score /100 | % récents | % doublons | étapes ADKAR | retenu |
|---|---|---|---|---|---|---|
| `"my job" AI` | 2 169 | 23 | 48 | 6 | AW DE KN AB | **oui** |
| `"my company" AI` | 1 470 | 29 | 49 | 6 | AW DE KN AB RE | **oui** |
| `"my team" AI` | 1 022 | 33 | 54 | 6 | AW DE KN AB RE | **oui** |
| `"my employer" AI` | 466 | 29 | 50 | 8 | AW DE KN AB RE | **oui** |
| `"my coworkers" AI` | 285 | 32 | 54 | 8 | AW DE KN AB | **oui** |
| `"my manager" AI` | 186 | 44 | 55 | 12 | AW DE KN AB RE | **oui** |
| `"my org" AI` | 170 | 37 | 59 | 7 | AW DE KN AB RE | non : précis mais petit volume |
| `"my workplace" AI` | 130 | 41 | 52 | 6 | AW DE KN AB RE | non : précis mais petit volume |
| `"at my job" AI` | 187 | 39 | 62 | 100 | AW DE AB | non : déjà inclus dans `"my job" AI` |
| `"my boss" AI` | 366 | 25 | 52 | 11 | AW DE KN AB RE | non : moins précis, pas de 2e passe |
| `"our company" AI` | 480 | 23 | 42 | 8 | AW DE KN AB RE | non : moins précis, pas de 2e passe |
| `copilot "my company"` | ≈ 170 | 45 | 100 | n.m. | AW DE KN AB RE | non : petit, limité à Copilot |
| `my manager AI` (sans guillemets) | ≈ 14 900 | 7 | 100 | n.m. | AW DE KN AB | non : bruit |
| `forced to use copilot` | ≈ 750 | 14 | 100 | n.m. | AW DE AB RE | non : bruit sans guillemets |
| `"forced to use copilot"` | 13 | 42 | 69 | n.m. | AW AB RE | non : volume quasi nul |
| `AI mandate` | ≈ 2 400 | 2 | 100 | n.m. | AW DE | non : débats généraux |
| `leadership pushing AI` | ≈ 330 | 26 | 100 | n.m. | AW DE KN AB RE | non : moins précis que les ancrages |
| `AI will replace my job` | ≈ 720 | 8 | 100 | n.m. | AW DE KN | non : spéculation, pas de vécu |
| `don't know how to use AI` | ≈ 27 700 | 2 | 100 | n.m. | KN | non : bruit |
| `stopped using copilot` | ≈ 180 | 10 | 62 | n.m. | AB RE | non : surtout usage perso |
| `company banned ChatGPT` | ≈ 40 | 5 | 18 | n.m. | AB | non : quasi vide |

### Justification du choix
- **Ce qui marche, c'est l'ancrage personnel** : une expression à la première personne entre guillemets (`"my manager"`, `"my team"`…) + le mot `AI`. Score moyen sur `/search_by_date` : **31/100** pour ces ancrages, contre **15** pour les phrases entières entre guillemets (volume quasi nul, médiane de 6 résultats) et **8** pour les mots libres (section 7). Les mots-clés pensés « par étape ADKAR » (`AI will replace my job`, `don't know how to use AI`, `gave up on AI tools`…) échouent tous : les gens ne décrivent pas leur blocage avec ces mots. Les étapes ADKAR apparaissent *dans* les témoignages ramenés par les ancrages.
- **Les guillemets sont respectés** par l'API (`advancedSyntax` est actif par défaut) : 90 à 100 % des résultats contiennent l'expression exacte, au pluriel, au possessif et à la ponctuation près.
- **`/search_by_date` plutôt que `/search`** : `/search` remonte ses 50 « meilleurs » résultats, trop optimistes (`my manager AI` sans guillemets fait 36/100 sur `/search`, mais 7/100 sur `/search_by_date`). Pour les ancrages entre guillemets, les deux endpoints trouvent le même ensemble : seul l'ordre change. `/search_by_date` donne donc une précision représentative de ce que l'ETL téléchargera, et permet une collecte incrémentale (« tout ce qui est plus récent que la dernière collecte »).
- **Les 6 retenus** : tous ont au moins 39 % de commentaires pertinents sur 70 lus en deux passes, et couvrent ensemble les 5 étapes. `"my job"`, `"my company"` et `"my team"` apportent le volume ; `"my manager"`, `"my coworkers"` et `"my employer"` apportent les témoignages les plus riches (le manager est le premier vecteur de l'imposition : 19 commentaires notés 2 sur 70).
- **Chevauchement faible** : 3,4 % de doublons dans l'union des 6, donc chaque mot-clé apporte des commentaires différents.

### Volume attendu
- **5 416 commentaires bruts uniques** depuis le 01/01/2023 ; **environ 1 700 pertinents estimés** (précision récente sur les 12 derniers mois, 19 % avant ; incertitude de l'ordre de ±25 %).
- Flux actuel : **environ 57 commentaires bruts par semaine, dont environ 25 pertinents**.
- **L'objectif de 2 000 n'est pas atteint d'emblée** : il manque environ 300 verbatims, soit environ 3 mois de collecte au rythme actuel. Il serait atteint immédiatement en ajoutant `"my org" AI`, `"my workplace" AI`, `"my boss" AI` et `"our company" AI` (environ +350 pertinents estimés), au prix d'une précision un peu plus faible et de plus de 6 mots-clés.

### Trois extraits représentatifs par mot-clé retenu (sans pseudo, tronqués)
**`"my job" AI`**
- [AW/AB] « Now management has decided to trust AI reviews, and we ship to production without a human in the loop. Now my job is to manage tokens and invent new ways of AI-guided testing of already shipped code after release. »
- [AW/DE] « For all my user facing work I'm now required to do an AI summary of what I did and the impact. That summary is then graded by AI, against an AI generated rubric. »
- [KN/AB] « My job is only to review and coach other junior/intermediate members of the team […] The problem with AI is the merge requests now contain more code and often the chosen solution is overblown/overdesigned. »

**`"my company" AI`**
- [AW/DE] « I have been raising this point in my company (which targets almost zero human code) but the answer is almost always about tooling and more AI-review. It's like talking to a wall. »
- [DE/AB] « My company laid off all our QA people and replaced them with AI. Let me tell you, a good QA person truly gives a shit. »
- [RE] « Reason why my company decided to stop using AI for code that gets delivered to customers, the customer find bugs, he is not happy, you have to use time to fix bugs. »

**`"my team" AI`**
- [AB] « This doesn't necessarily make my team any more effective overall. It seems like it's been a code review burden and it's led to some team members being too ambitious with picking up more projects. »
- [DE] « The backlog has completely emptied and some of my team will be surplus to requirements. »
- [RE] « This currently is my team's only AI-use policy and I thik [sic] it's working out fantastically. "No AI PR Descriptions" is a great rule. »

**`"my employer" AI`**
- [AW/AB] « My employer has been integrating agentic AI into every tool and updating the UI/UX to make the agent the primary interface. This applies everywhere, including people management tasks. It sucks. »
- [AB] « My employer won't even provide LLMs for us, let alone allow us to use agentic coding on our repos. »
- [AB/RE] « My employer already introduced a hard cap on AI credits while rolling out hardware support for local models. »

**`"my coworkers" AI`**
- [AW/AB] « Many of my coworkers are in a state of unreality that seems related to AI usage. […] I use them too, not least because it's expected of me by management. »
- [AB] « My coworkers would copy and paste Claude-written emails into Gmail, and their response rate was 0.005%-0.05%. »
- [DE/AB] « I'm still looking for a way to leave tech because of it. […] internal teams not getting properly supported because they aren't on the "AI first team or division". »

**`"my manager" AI`**
- [AW/DE/RE] « My manager preferred to fix forward rather than address my concerns that we were producing tech debt at 10x speeds, so none of my comments were addressed. »
- [AB] « My manager uses Claude for EVERYTHING and is completely careless with it. Hallucinations in performance reviews. »
- [KN/AB] « I've been training my co-workers that the AI-isms their "AI Assisted coding" do that some shit just isn't acceptable. […] My manager is also 100% with this. »

### Limites connues (à dire au jury)
- **Biais de population** : Hacker News, c'est surtout des développeurs anglophones, souvent nord-américains et travaillant dans la tech, plutôt critiques envers l'IA. Peu de non-techniques, peu d'Europe non anglophone, presque pas de petites entreprises hors tech.
- **Biais de sujet** : l'ancrage « my X + AI » privilégie ceux qui *parlent de leur entourage* ; les vécus purement individuels (« j'ai essayé, j'ai abandonné ») sont moins captés. **Knowledge (KN) et Reinforcement (RE) sont sous-représentés** (19 et 12 occurrences sur 70 × 6 commentaires lus, contre 84 pour Ability). Les mots-clés ciblés sur ces étapes ont été testés et ne fonctionnent pas.
- **Notation par une seule personne** (pas de double codage, donc pas d'accord inter-annotateurs mesuré) ; notes 0/1 parfois discutables pour les témoignages rapportés (« un ami me dit que… »), comptés 1.
- **Précision variable dans le temps** : environ 50 % en 2026, environ 19 % en 2023-2024 (sur seulement 70 commentaires anciens). L'estimation du volume pertinent est donc fragile.
- **`nbHits` n'est pas fiable** : c'est parfois une estimation qui change d'un appel à l'autre (`exhaustive.nbHits = False`) ; il faut compter les résultats réellement téléchargés.
- **Plafond de 1 000 résultats par requête** : obligation de découper la collecte par tranches de dates.
- L'index Algolia évolue en continu : les chiffres changent légèrement d'une exécution à l'autre si on vide le cache.

### Problèmes de données repérés pour le nettoyage (section 12)
- **HTML** dans `comment_text` : `<p>` (85 % des commentaires), liens `<a href>` (14 %), `<i>`, `<pre><code>` (19 %).
- **Entités HTML** : `&#x27;` (apostrophe), `&quot;`, `&gt;`, `&amp;`, `&#x2F;` (84 %) ; utiliser `html.unescape`. Apostrophes typographiques `’` à normaliser.
- **Citations** d'un autre commentaire en début de ligne (`&gt;` puis texte, 18 %) : à retirer avant de classer, sinon on attribue à l'auteur les mots d'un autre.
- **Pseudos** : champ `author` à supprimer (RGPD, et consigne). Mentions dans le texte (`@xxx`, « GP », « parent ») rares (1 %).
- **Doublons** : même commentaire via plusieurs mots-clés (dédoublonner sur `objectID`) et, plus rare, **même texte sous deux `objectID`** (19 cas sur 4 458) : ajouter un dédoublonnage sur le texte normalisé.
- **Textes courts** : environ 4 % des commentaires font moins de 150 caractères une fois nettoyés.
- **Correspondances sur le titre du fil** et non sur le commentaire : à éviter avec `restrictSearchableAttributes=comment_text`.
- **Bruit restant** : fils d'offres d'emploi (« Who is hiring / Seeking work ») et ironie ou satire (fausses histoires de CEO, copypastas) à filtrer ou à signaler au classement.